In [1]:
T = CartanType(['C', 3])
W = WeylGroup(T,prefix="s")
[s1, s2, s3] = W.simple_reflections()

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 7

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KL.P(y, w)(1)) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KL.P(y, w0*w)(1)) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [2]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sub[i], d_sup[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [3]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sup[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sub[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sub[i], d_dual[j])*d_sup[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [4]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sub[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

Checked dual basis, all is good.


In [5]:
#Defining [q]^*f_w
q = 29

d_qsub = {}
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_qsub[i] = qfsub(W[i])

In [6]:
#DEPENDS ON THE TYPE! Set the minimal Duflo involution in each left cell. Leave lcell_dict = {} in Type A!
lcell_dict = {s1^2 : s1^2, s1 : s1, s1*s2*s3*s2*s1 : s1,
              s2 : s2, s2*s3*s2 : s2,
              s3 : s3, s3*s2*s3 : s3,
              s3*s1 : s3*s1, s2*s3*s1*s2 : s2*s3*s1*s2, s3*s2*s3*s1*s2*s3 : s3*s2*s3*s1*s2*s3,
              s1*s2*s1 : s1*s2*s1, s3*s1*s2*s3*s1 : s3*s1*s2*s3*s1, s2*s3*s1*s2*s3*s1*s2 : s2*s3*s1*s2*s3*s1*s2,
              s3*s2*s3*s2 : s3*s2*s3*s2, s3*s2*s3*s1*s2*s3*s1*s2 : s3*s2*s3*s2,
              s3*s1*s2*s3*s2*s1 : s3*s1*s2*s3*s2*s1, s3*s2*s3*s1*s2*s3*s2*s1 : s3*s1*s2*s3*s2*s1,
              s1*s2*s3*s1*s2*s1 : s1*s2*s3*s1*s2*s1, s2*s3*s1*s2*s3*s1*s2*s1 : s1*s2*s3*s1*s2*s1,
              w0 : w0}
for a in invols:
    if a not in lcell_dict:
        lcell_dict[a] = a
print("lcell_dict is defined.")
print(lcell_dict)

lcell_dict is defined.
{1: 1, s1: s1, s1*s2*s3*s2*s1: s1, s2: s2, s2*s3*s2: s2, s3: s3, s3*s2*s3: s3, s3*s1: s3*s1, s2*s3*s1*s2: s2*s3*s1*s2, s3*s2*s3*s1*s2*s3: s3*s2*s3*s1*s2*s3, s1*s2*s1: s1*s2*s1, s3*s1*s2*s3*s1: s3*s1*s2*s3*s1, s2*s3*s1*s2*s3*s1*s2: s2*s3*s1*s2*s3*s1*s2, s3*s2*s3*s2: s3*s2*s3*s2, s3*s2*s3*s1*s2*s3*s1*s2: s3*s2*s3*s2, s3*s1*s2*s3*s2*s1: s3*s1*s2*s3*s2*s1, s3*s2*s3*s1*s2*s3*s2*s1: s3*s1*s2*s3*s2*s1, s1*s2*s3*s1*s2*s1: s1*s2*s3*s1*s2*s1, s2*s3*s1*s2*s3*s1*s2*s1: s1*s2*s3*s1*s2*s1, s3*s2*s3*s1*s2*s3*s1*s2*s1: s3*s2*s3*s1*s2*s3*s1*s2*s1}


In [7]:
def nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(lcell_dict[y])])

table = []

Mw = {}

for w in invols:
    a = woke_pairing(d_qsub[list(W).index(w0*w)], d_dual[list(W).index(lcell_dict[w0*w])])
    Mw[w] = a
    print("$" + str(w) + "$", "&", "$" + str(a) + "$ \\\\")

$1$ & $C3(0,0,0)$ \\
$s3$ & $-C3(0,1,26) + C3(1,0,27)$ \\
$s2$ & $C3(0,25,0) - C3(1,25,1) + C3(2,26,0)$ \\
$s2*s3*s2$ & $C3(2,25,0) - C3(1,25,1) + C3(0,28,0)$ \\
$s2*s3*s1*s2$ & $-C3(0,25,0) + C3(0,26,0) - C3(2,25,0) - C3(0,27,0) + C3(2,26,0) + C3(0,28,0)$ \\
$s3*s2*s3*s1*s2*s3*s1*s2*s1$ & $C3(28,28,28)$ \\
$s2*s3*s1*s2*s3*s1*s2*s1$ & $C3(28,26,0) + C3(28,27,0) + C3(28,28,0)$ \\
$s3*s2*s3*s1*s2*s3*s2*s1$ & $C3(26,0,28) + C3(27,1,27) + C3(28,0,28)$ \\
$s3*s1*s2*s3*s2*s1$ & $C3(27,0,27) + C3(27,1,27) + C3(29,0,27)$ \\
$s3*s1*s2*s3*s1$ & $-C3(27,0,27) - C3(26,0,28) + C3(29,0,27) + C3(28,0,28)$ \\
$s1$ & $C3(24,0,0) + C3(26,0,0)$ \\
$s3*s1$ & $C3(27,0,27) - C3(27,1,27) + C3(28,0,28)$ \\
$s1*s2*s3*s2*s1$ & $C3(26,0,0) + C3(28,0,0)$ \\
$s1*s2*s1$ & $-C3(28,26,0) + C3(28,28,0)$ \\
$s1*s2*s3*s1*s2*s1$ & $C3(26,28,0) + C3(28,27,0) + C3(30,26,0)$ \\
$s3*s2*s3*s1*s2*s3*s1*s2$ & $C3(0,28,28)$ \\
$s2*s3*s1*s2*s3*s1*s2$ & $C3(0,25,0) + C3(0,26,0) + C3(1,25,1) + C3(0,27,0) + C3(0,28,0)$ \\
$s3*s2*s3*

In [8]:
print(w0*s3*s2*s3*s1*s2*s3*s1*s2)

s1
